In [1]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
# Permite importar `proyecto` sin instalarlo: el código vive en src/
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))

# 08 · Registro del modelo

En la etapa 6 hubo **muchas ejecuciones**, una por ensayo: eso es **tracking**. Ahora hay **un modelo final**, y lo que se
guarda es otra cosa: una **versión del modelo**, con su firma, su ejemplo de entrada, sus métricas de la prueba y un
**alias** (`champion`) que dice cuál es el vigente.

| | Tracking | Registro |
|---|---|---|
| Qué guarda | Cada ensayo: parámetros y métricas | El modelo final, versionado |
| Cuántos hay | Muchos | Pocos |
| Para qué | Comparar y reproducir | Saber **cuál** es el modelo en uso y poder volver a una versión anterior |

> **Seguridad:** un archivo *pickle* **puede ejecutar código al cargarse**. Solo se cargan modelos que registró el grupo.

## 0. Conexión con el seguimiento

Dentro de Databricks no hay que configurar nada. Desde el computador se lee `.env`. Si no hay conexión, o se agotó la
cuota, se usa MLflow **local** (`mlflow_local.db`), que es el respaldo previsto por el curso.

In [2]:
import json
from datetime import datetime

import joblib
import mlflow
import numpy as np
import pandas as pd
from mlflow import MlflowClient
from mlflow.models import infer_signature

from proyecto.config import ANONYMOUS_GROUP_COL, MODELS_DIR, PROCESSED_DIR, REPORTS_DIR, TARGET
from proyecto.inference.predict import input_frame
from proyecto.tracking.mlflow_utils import configure_mlflow, registered_model_name, set_experiment

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    where = configure_mlflow()
for w in caught:
    print("Aviso:", w.message)
experiment = set_experiment("eco-score-registro")
MODEL_NAME = registered_model_name("eco_score_default")
print("MLflow:", where)
print("Nombre del modelo en el registro:", MODEL_NAME)

2026/11/05 20:25:21 INFO mlflow.tracking.fluent: Experiment with name 'eco-score-registro' does not exist. Creating a new experiment.


MLflow: local
Nombre del modelo en el registro: eco_score_default


## 1. Cargar el modelo final y comprobar que es el evaluado

El pipeline lo dejó el notebook 07 en `models/final_pipeline.joblib`. Ese archivo **no se versiona**: si falta, se vuelve a
ejecutar el notebook 07, que no reevalúa la prueba.

**Comprobación antes de registrar:** el modelo debe dar **las mismas probabilidades** que se guardaron al abrir la prueba.
Aquí no se calcula ninguna métrica ni se decide nada: solo se comparan dos listas de probabilidades para confirmar que
se registra el modelo evaluado.

In [3]:
model_path = MODELS_DIR / "final_pipeline.joblib"
if not model_path.exists():
    raise FileNotFoundError("Falta models/final_pipeline.joblib: ejecuta antes el notebook 07.")
final_model = joblib.load(model_path)
results = json.loads((REPORTS_DIR / "07_metricas_test.json").read_text(encoding="utf-8"))
decisions = results["decisiones"]
print("modelo:", decisions["modelo_final"], "· calibrado:", decisions["calibrar"])
print("decisiones congeladas el:", decisions["fecha_y_hora"])

train = pd.read_parquet(PROCESSED_DIR / "train_sin_memberkey.parquet")
test = pd.read_parquet(PROCESSED_DIR / "test_sin_memberkey.parquet")
evaluated = joblib.load(PROCESSED_DIR / "prueba_probabilidades.joblib")["final"]
inputs_train, inputs_test = input_frame(train), input_frame(test)
same_model = bool(np.allclose(final_model.predict_proba(inputs_test), evaluated))
assert same_model, "El modelo no da las probabilidades evaluadas en la etapa 7: no se registra."
assert not {"MemberKey", ANONYMOUS_GROUP_COL, TARGET} & set(inputs_train.columns)

example = inputs_train.head(200)
print("es el modelo evaluado:", same_model)
print("firma sobre", len(example), "filas de ejemplo,", example.shape[1], "columnas de entrada")

modelo: lightgbm · calibrado: False
decisiones congeladas el: 2026-11-05T20:09:49


es el modelo evaluado: True
firma sobre 200 filas de ejemplo, 43 columnas de entrada


## 2. Qué se registra

Junto al modelo viajan:

- los **parámetros** y las **decisiones**: qué modelo es, si se calibró y con qué regla decide;
- las **métricas de la prueba**, incluida la del tramo desde julio de 2009;
- la **firma** (columnas y tipos de entrada, probabilidades de salida) y un **ejemplo de entrada**;
- **etiquetas** para encontrarlo después.

Las **columnas de entrada** son las crudas que requiere el pipeline. No incluyen `MemberKey`, `borrower_id` ni `BorrowerState`.

> **Nota técnica:** hay que pasar `serialization_format="cloudpickle"`. El formato por defecto de MLflow rechaza guardar un
> `Pipeline` con transformadores propios (error *untrusted types*).

In [4]:
intervals = {row["medida"]: row for row in results["intervalos"]}
pr_interval = intervals["AUC-PR (mayor es mejor)"]
recent = results["desde_julio_2009"]
test_metrics = {f"test_{key}": float(value) for key, value in results["metricas_modelo"].items()}
test_metrics.update({
    "test_pr_auc_ci_low": pr_interval["IC 95 % inferior"], "test_pr_auc_ci_high": pr_interval["IC 95 % superior"],
    "test_pr_auc_baseline": results["metricas_linea_base"]["pr_auc"],
    "test_pr_auc_since_2009": recent["pr_auc"], "test_pr_auc_since_2009_ci_low": recent["ic95"][0],
    "test_pr_auc_since_2009_ci_high": recent["ic95"][1],
})
signature = infer_signature(example, final_model.predict_proba(example))

with mlflow.start_run(run_name="modelo-final") as run:
    mlflow.log_params({
        "modelo": decisions["modelo_final"], "calibrado": decisions["calibrar"], "umbral_de_costo": 0.2,
        **{f"param_{key}": value for key, value in decisions["parametros"].items()},
    })
    mlflow.log_metrics(test_metrics)
    mlflow.set_tags({
        "etapa": "08_registro", "decisiones_congeladas": decisions["fecha_y_hora"],
        "unidad_de_particion": "prestatario", "prueba_filas": results["prueba"]["prestamos"],
        "supera_linea_base": str(results["supera_a_la_linea_base_con_confianza"]),
    })
    logged = mlflow.sklearn.log_model(
        final_model, name="model", signature=signature, input_example=example.head(5),
        serialization_format="cloudpickle", registered_model_name=MODEL_NAME,
    )
    run_id = run.info.run_id

client = MlflowClient()
version = max(int(v.version) for v in client.search_model_versions(f"name='{MODEL_NAME}'"))
print("ejecución:", run_id)
print(f"modelo registrado: {MODEL_NAME} · versión {version}")

2026/11/05 20:25:23 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


2026/11/05 20:25:37 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


ejecución: 94e6c9a0a24744ac8d3220766a381e57
modelo registrado: eco_score_default · versión 1


Successfully registered model 'eco_score_default'.
Created version '1' of model 'eco_score_default'.


## 3. Poner el alias `champion`

Una **versión** es un número que no dice nada sobre su uso. El **alias** sí: `champion` es "el modelo vigente". Quien use
el modelo pide `models:/<nombre>@champion` y no necesita saber qué número de versión es. Si mañana hay una versión mejor,
se mueve el alias y nadie cambia su código; si algo sale mal, se mueve de vuelta.

In [5]:
client.set_registered_model_alias(MODEL_NAME, "champion", str(version))
client.set_model_version_tag(MODEL_NAME, str(version), "decisiones_congeladas", decisions["fecha_y_hora"])
champion = client.get_model_version_by_alias(MODEL_NAME, "champion")
print(f"alias champion -> versión {champion.version}  (ejecución {champion.run_id[:8]}…)")

alias champion -> versión 1  (ejecución 94e6c9a0…)


## 4. Comprobación de ida y vuelta

Se **carga desde el registro** por el alias y se comprueba que predice **exactamente igual** que el objeto registrado.
Si no, lo que está en el registro no es lo que se evaluó.

In [6]:
from_registry = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}@champion")
rows = inputs_train.head(500)
expected, obtained = final_model.predict_proba(rows), from_registry.predict_proba(rows)
round_trip = bool(np.allclose(expected, obtained))
print("las 500 predicciones coinciden:", round_trip, f"(diferencia máxima {np.abs(expected - obtained).max():.2e})")
assert round_trip, "El modelo del registro no coincide con el que se evaluó"

las 500 predicciones coinciden: True (diferencia máxima 0.00e+00)


## 5. Guardar la constancia

Queda en `reports/08_registro.json` y se resume en `reports/08_registro.md`. **La app no consulta el registro**: trabaja
con el `model.joblib` que se exporta en el notebook 09.

In [7]:
record = {
    "fecha_y_hora": datetime.now().isoformat(timespec="seconds"), "donde": where,
    "experimento": experiment.rsplit("/", 1)[-1],   # sin la carpeta del usuario, que incluye su correo en Databricks
    "modelo_registrado": MODEL_NAME, "version": version, "alias": "champion", "ejecucion": run_id,
    "es_el_modelo_evaluado": same_model, "ida_y_vuelta_ok": round_trip, "modelo": decisions["modelo_final"],
    "calibrado": decisions["calibrar"], "decisiones_congeladas": decisions["fecha_y_hora"],
    "columnas_de_entrada": list(example.columns),
}
(REPORTS_DIR / "08_registro.json").write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps({k: v for k, v in record.items() if k != "columnas_de_entrada"}, indent=2, ensure_ascii=False))

{
  "fecha_y_hora": "2026-11-05T20:25:38",
  "donde": "local",
  "experimento": "eco-score-registro",
  "modelo_registrado": "eco_score_default",
  "version": 1,
  "alias": "champion",
  "ejecucion": "94e6c9a0a24744ac8d3220766a381e57",
  "es_el_modelo_evaluado": true,
  "ida_y_vuelta_ok": true,
  "modelo": "lightgbm",
  "calibrado": false,
  "decisiones_congeladas": "2026-11-05T20:09:49"
}


## Qué sigue

**Notebook 09:** exportar `models/model.joblib` desde `champion`, escribir el esquema de entrada y comprobar que la función
de predicción rechaza los datos imposibles. Con eso la app ya puede cargar el modelo.